<a href="https://colab.research.google.com/github/AsfandyarFarooqi/learning-log/blob/main/week07_real_gnss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
np.set_printoptions(suppress=True)

In [2]:
a = 6378137.0
f = 1 / 298.257223563
e2 = 2 * f - f**2
b = a * (1 - f)
print("f =", round(f, 10), "| e2 =", round(e2, 10), "| b =", round(b, 3), "m")

def geodetic_to_ecef(lat_deg, lon_deg, h):
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    N = a / np.sqrt(1 - e2 * np.sin(lat)**2)
    X = (N + h) * np.cos(lat) * np.cos(lon)
    Y = (N + h) * np.cos(lat) * np.sin(lon)
    Z = (N * (1 - e2) + h) * np.sin(lat)
    return np.array([X, Y, Z])

test = geodetic_to_ecef(30, 60, 0)
print("test point 30 N, 60 E, h = 0 (m):", np.round(test, 1))
p = np.sqrt(test[0]**2 + test[1]**2)
print("p =", round(p, 1), "m | on the ellipse? (p/a)^2 + (Z/b)^2 =", round((p / a)**2 + (test[2] / b)**2, 10))

f = 0.0033528107 | e2 = 0.00669438 | b = 6356752.314 m
test point 30 N, 60 E, h = 0 (m): [2764128.3 4787610.7 3170373.7]
p = 5528256.6 m | on the ellipse? (p/a)^2 + (Z/b)^2 = 1.0


In [3]:
truth_ecef = geodetic_to_ecef(37.692231, -122.0884199, 20.974)
print("phone truth in ECEF (m):", np.round(truth_ecef, 3))
print("distance from Earth's centre (km):", round(np.sqrt(np.sum(truth_ecef**2)) / 1000, 3))

phone truth in ECEF (m): [-2684506.844 -4281392.596  3878481.691]
distance from Earth's centre (km): 6370.206


In [4]:
def enu_matrix(lat_deg, lon_deg):
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    east = [-np.sin(lon), np.cos(lon), 0.0]
    north = [-np.sin(lat) * np.cos(lon), -np.sin(lat) * np.sin(lon), np.cos(lat)]
    up = [np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)]
    return np.array([east, north, up])

R_test = enu_matrix(30, 60)
print("R at 30 N, 60 E:\n", np.round(R_test, 4))
print("R @ R.T:\n", np.round(R_test @ R_test.T, 10) + 0.0)
print("move (0, 0, 100) as E, N, U:", np.round(R_test @ np.array([0, 0, 100]), 2))
print("move (100, 0, 0) as E, N, U:", np.round(R_test @ np.array([100, 0, 0]), 2))

R at 30 N, 60 E:
 [[-0.866  0.5    0.   ]
 [-0.25  -0.433  0.866]
 [ 0.433  0.75   0.5  ]]
R @ R.T:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
move (0, 0, 100) as E, N, U: [ 0.  86.6 50. ]
move (100, 0, 0) as E, N, U: [-86.6 -25.   43.3]


In [5]:
wls_ecef = np.array([-2684512.903, -4281393.666, 3878486.752])
d_ecef = wls_ecef - truth_ecef
R_phone = enu_matrix(37.692231, -122.0884199)
d_enu = R_phone @ d_ecef
print("Google's answer minus truth, ECEF (m):", np.round(d_ecef, 2))
print("Google's answer minus truth, E, N, U (m):", np.round(d_enu, 2))
print("length in ECEF:", round(np.sqrt(np.sum(d_ecef**2)), 2), "m | length in ENU:", round(np.sqrt(np.sum(d_enu**2)), 2), "m")

Google's answer minus truth, ECEF (m): [-6.06 -1.07  5.06]
Google's answer minus truth, E, N, U (m): [-4.56  1.48  6.36]
length in ECEF: 7.97 m | length in ENU: 7.97 m


In [6]:
sat_G10 = np.array([-5111739.857, -13974794.020, 22176382.208])
enu_G10 = R_phone @ (sat_G10 - truth_ecef)
r_G10 = np.sqrt(np.sum(enu_G10**2))
el_G10 = np.degrees(np.arcsin(enu_G10[2] / r_G10))
az_G10 = np.degrees(np.arctan2(enu_G10[0], enu_G10[1]))
print("G10 as E, N, U (km):", np.round(enu_G10 / 1000, 3))
print("range (km):", round(r_G10 / 1000, 3))
print("elevation:", round(el_G10, 2), "deg | azimuth:", round(az_G10, 2), "deg")

G10 as E, N, U (km): [ 3092.977  8669.569 18706.649]
range (km): 20848.661
elevation: 63.8 deg | azimuth: 19.63 deg


In [7]:
sat_G2=np.array([-14916644.088,8381795.848,20772371.222])
enu_G2=R_phone@(sat_G2-truth_ecef)
r_G2=np.sqrt(np.sum(enu_G2**2))
el_G2=np.degrees(np.arcsin(enu_G2[2]/r_G2))
az_G2=np.degrees(np.arctan2(enu_G2[0],enu_G2[1]));
print("G2 as E, N, U (km):",np.round(enu_G2/1000,3))
print("range (km):",round(r_G2/1000,3))
print("elevation:",round(el_G2,2),"deg | azimuth:",round(az_G2,2),"deg")

G2 as E, N, U (km): [-17090.457  15954.903   6981.574]
range (km): 24400.513
elevation: 16.63 deg | azimuth: -46.97 deg


In [8]:
svid = np.array([2, 8, 10, 18, 21, 23, 24, 27, 28, 32])
sats_ecef = np.array([[-14916644.088, 8381795.848, 20772371.222],
                      [-20987738.292, -1376793.804, 16379998.400],
                      [-5111739.857, -13974794.020, 22176382.208],
                      [4662547.027, -26046772.507, -304155.421],
                      [-15329996.983, 4873323.738, 21506733.093],
                      [8735732.507, -17141539.403, 18238905.248],
                      [14512036.477, -11225369.117, 18680721.785],
                      [-22804233.674, -12220838.375, 6640973.669],
                      [-6555214.310, -22205195.468, -13039139.727],
                      [-14858588.220, -20090152.111, 9473949.458]])
raw = np.array([24567440.915, 22397058.939, 20864402.974, 23472434.871, 23502523.242,
                22386395.921, 23822439.568, 21815989.505, 24951022.966, 20888952.079])
sat_clock = np.array([-166876.028, -42304.128, -15721.262, -122784.859, 44763.811,
                      19570.306, -93804.429, -10684.238, -2019.730, -166129.923])
iono = np.array([9.277, 6.805, 4.827, 10.145, 8.076, 7.199, 9.803, 6.232, 15.365, 4.905])
tropo = np.array([8.645, 4.165, 2.784, 6.487, 5.885, 4.434, 9.052, 3.519, 17.993, 2.706])

rho = raw + sat_clock - iono - tropo
print("satellites:", svid)
print("corrected pseudoranges (m):", np.round(rho, 3))

satellites: [ 2  8 10 18 21 23 24 27 28 32]
corrected pseudoranges (m): [24400546.965 22354743.841 20848674.101 23349633.38  23547273.092
 22405954.594 23728616.284 21805295.516 24948969.878 20722814.545]


In [9]:
def gnss_3d(sats, rho, x0, stop=0.0001, max_rounds=10, show=True):
    x = np.array(x0, dtype=float)
    n = len(rho)
    for round_no in range(1, max_rounds + 1):
        diff = x[:3] - sats
        d0 = np.sqrt(np.sum(diff**2, axis=1))
        A = np.ones((n, 4))
        A[:, :3] = diff / d0.reshape(n, 1)
        drho = rho - (d0 + x[3])
        dx = np.linalg.solve(A.T @ A, A.T @ drho)
        x = x + dx
        size = np.sqrt(np.sum(dx**2))
        if show:
            print("round", round_no, "| size of move:", round(size, 3), "m | X, Y, Z, b:", np.round(x, 3))
        if size < stop:
            break
    return x

x_hat = gnss_3d(sats_ecef, rho, np.array([0.0, 0.0, 0.0, 0.0]))

round 1 | size of move: 7572777.281 m | X, Y, Z, b: [-3166453.962 -5042663.619  4524330.895  1192685.47 ]
round 2 | size of move: 1588721.412 m | X, Y, Z, b: [-2695811.359 -4299439.967  3892254.125    30519.894]
round 3 | size of move: 39641.377 m | X, Y, Z, b: [-2684501.162 -4281420.862  3878485.227       36.333]
round 4 | size of move: 21.552 m | X, Y, Z, b: [-2684494.743 -4281410.737  3878478.008       19.943]
round 5 | size of move: 0.0 m | X, Y, Z, b: [-2684494.743 -4281410.737  3878478.008       19.943]


In [10]:
err_enu = R_phone @ (x_hat[:3] - truth_ecef)
print("your error E, N, U (m):", np.round(err_enu, 2), "| length:", round(np.sqrt(np.sum(err_enu**2)), 2), "m")
print("Google's error E, N, U (m):", np.round(d_enu, 2), "| length:", round(np.sqrt(np.sum(d_enu**2)), 2), "m")

your error E, N, U (m): [19.89 -8.38  4.82] | length: 22.12 m
Google's error E, N, U (m): [-4.56  1.48  6.36] | length: 7.97 m
